In [ ]:
import pyGinkgo as pg

### Direct solver bindings

In [ ]:
fn = "m1.mtx"
dev = pg.device("cuda")
mtx = pg.read(path=fn, dtype="double", format="Csr")
n_rows = mtx.size[0]

b = pg.as_tensor(device=dev, dim=(n_rows, 1), dtype="double", fill=1.0)
x = pg.as_tensor(device=dev, dim=(n_rows, 1), dtype="double", fill=0.0)

# Create ILU preconditioner
preconditioner = pg.preconditioner.Ilu(dev, mtx)

# Setup GMRES solver
solver = pg.solver.gmres(
    dev,
    mtx,
    preconditioner=preconditioner,
    max_iters=1000,
    krylov_dim=30,
    reduction_factor=1e-06,
)

# Apply
logger, result = solver.apply(b, x)

In [ ]:
result_cpu = result.copy_to_host()

In [ ]:
for i in range(10):
    print(result_cpu.at(i, 0), end=", ")

### Config solver

In [ ]:
args = {
    "type": "solver::Gmres",
    "krylov_dim": 30,
    "preconditioner": {
        "type": "preconditioner::Jacobi",
        "max_block_size": 1
    },
    "criteria": [
        {"type": "Iteration", "max_iters": 1000},
        {
          "type": "ResidualNorm", 
          "reduction_factor": 1e-6, 
          "baseline": "rhs_norm"
        }
    ],
}

In [ ]:
b = pg.as_tensor(device=dev, dim=(n_rows, 1), dtype="double", fill=1.0)
x = pg.as_tensor(device=dev, dim=(n_rows, 1), dtype="double", fill=0.0)

solver = pg.generate_solver(mtx, args)

logger, result = solver.apply(b, x)

In [ ]:
result_cpu_2 = result.copy_to_host()

### Checking difference between two solvers

In [ ]:
result_cpu_2.add_scaled(
    pg.as_tensor(device=dev, dim=(1, 1), fill=-1),
    result_cpu
)

In [ ]:
for i in range(10):
    print(result_cpu_2.at(i, 0), end=", ")